# Stress Prediction — FINAL (v34 원본 로직, 최종 제출용)

오늘 시도한 11가지 실험(도메인 피처, allostatic load, SVR, LightGBM x3, RandomForest,
pair k-NN, FEATURE_COPY_COUNTS 재설계, PAIR_COLUMNS 확장, activity 상호작용) 전부
v34(CV MAE 0.148033 / Public MAE 0.127187)를 넘지 못했습니다.

**이 노트북은 v34의 검증된 로직을 그대로 재현한 최종 제출용입니다.** 새로운 시도 없이,
`validate_inputs`로 제출 형식만 한 번 더 안전하게 체크하고 바로 제출 파일을 만듭니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")     # train/test/sample_submission 폴더
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")    # 제출 파일 저장 폴더
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


In [2]:

def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


def predict_final(train_features, val_features, target) -> np.ndarray:
    tree = tree_prediction(train_features, val_features, target)
    pair = pair_prediction(train_features, val_features, target)
    return np.clip(np.round((1 - PAIR_WEIGHT) * tree + PAIR_WEIGHT * pair, 2), 0.0, 1.0)


def validate_inputs(train: pd.DataFrame, test: pd.DataFrame, submission: pd.DataFrame) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")


## 1. (선택) 5-Fold CV로 최종 sanity check

시간 여유가 있으면 실행해서 CV MAE가 0.148 근처인지 확인하세요. 급하면 건너뛰고 바로 2번으로 가도 됩니다.


In [3]:

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
validate_inputs(train, test, submission)

X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof = np.zeros(len(y))
for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):
    oof[val_idx] = predict_final(X.iloc[train_idx], X.iloc[val_idx], y.iloc[train_idx])
    fold_mae = mean_absolute_error(y.iloc[val_idx], oof[val_idx])
    print(f"Fold {fold} MAE: {fold_mae:.6f}")

print("CV MAE:", f"{mean_absolute_error(y, oof):.6f}", "(참고: v34 원본 0.148033 근처면 정상)")


Fold 1 MAE: 0.133900
Fold 2 MAE: 0.150017
Fold 3 MAE: 0.147633
Fold 4 MAE: 0.157783
Fold 5 MAE: 0.150833
CV MAE: 0.148033 (참고: v34 원본 0.148033 근처면 정상)


## 2. 최종 제출 파일 생성 (전체 train으로 재학습)

In [4]:

final_prediction = predict_final(X, X_test, y)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_final_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
print("Row count:", len(output), "(test.csv와 같아야 함:", len(test), ")")
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_final_0806.csv
Prediction range: 0.0 1.0
Row count: 3000 (test.csv와 같아야 함: 3000 )


,ID,stress_score
0,TEST_0000,0.59
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.50
4,TEST_0004,0.53
5,TEST_0005,0.45
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.58
9,TEST_0009,0.88
